# HiGAN+ — Train on Kaggle (T4 / P100)

Fine-tune HiGAN+ on the IAM word dataset starting from the upstream `deploy_HiGAN+.pth` checkpoint.

**Requirements before running:**
- Notebook *Settings* → **Accelerator: GPU T4 / P100** and **Internet: ON**.
- ~1 GB free disk for IAM hdf5 + pretrained checkpoints.

**Outputs land under `/kaggle/working/higanplus/HiGAN+/runs/`** — Kaggle persists `/kaggle/working/` as the notebook output, so checkpoints and tensorboard logs survive after the session ends.

Pipeline:
1. Clone fork → install deps (skip torch, Kaggle's image already has CUDA torch).
2. Download IAM hdf5 + 4 pretrained `.pth` from upstream releases.
3. Smoke-run (2 epochs × 30 iters) to verify the loop on Kaggle hardware.
4. Full fine-tune: 70 epochs from `gan_iam_kaggle.yml` (linear LR decay from epoch 25).

## 1. Environment check

In [ ]:
import torch, sys
print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__)
print('cuda    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    print('cuda ver:', torch.version.cuda)

## 2. Clone repo (fork with CUDA-12.x / PyTorch 2.x compat fixes)

In [ ]:
%cd /kaggle/working
![ -d higanplus ] || git clone --depth 1 https://github.com/ks2n/higanplus.git
%cd /kaggle/working/higanplus
!git log -1 --oneline

## 3. Install Python deps
Kaggle ships a CUDA-enabled torch already, so `requirements.txt` skips torch.

In [ ]:
!pip install -q -r requirements.txt

## 4. Download IAM dataset + pretrained checkpoints
Idempotent: `setup_data.sh` skips files that already exist. Total ~430 MB.

In [ ]:
!bash scripts/setup_data.sh

## 5. Smoke run (2 epochs × 30 iters, batch=4)
Verifies dataloader + model + losses + checkpoint write before committing to a 70-epoch run.
Takes ~1–2 minutes on a T4.

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_smoke.yml

In [ ]:
# Inspect smoke outputs
import glob, os
smoke_runs = sorted(glob.glob('/kaggle/working/higanplus/HiGAN+/runs/gan_iam_smoke-*'))
print('smoke runs:', smoke_runs)
if smoke_runs:
    last = smoke_runs[-1]
    print('files in last run:')
    for p in sorted(glob.glob(os.path.join(last, '**'), recursive=True))[:20]:
        print(' ', p)

# Show a sample image if produced
from IPython.display import Image, display
imgs = sorted(glob.glob(os.path.join(smoke_runs[-1], 'imgs', '*.png'))) if smoke_runs else []
if imgs:
    display(Image(filename=imgs[-1]))

## 6. Full training — 70 epochs, fine-tune from `deploy_HiGAN+.pth`

Config: `configs/gan_iam_kaggle.yml`
- batch_size: 16, eval_batch_size: 16
- epochs: 70, linear LR decay from epoch 25 (over 46 epochs)
- starts from `pretrained/deploy_HiGAN+.pth` so it converges fast
- saves checkpoint every epoch (`save_epoch_val: 1`) under `runs/gan_iam_kaggle-<timestamp>/ckpts/`
- writes sample images every 500 iters

**Time estimate on Kaggle T4:** ~6–10 min/epoch → 70 epochs ≈ 7–11h.
Kaggle limit is 12h/session, so this fits in one run, but resume support is shown below in case you stop early.

Tensorboard logs are written to the same run dir; you can launch tensorboard in another cell with `%load_ext tensorboard` + `%tensorboard --logdir runs`.

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_kaggle.yml

## 7. (Optional) Resume from a previous run

If the 12h limit cut training short, a `last.pth` was saved every epoch under
`runs/gan_iam_kaggle-<timestamp>/ckpts/last.pth`. To resume, edit
`configs/gan_iam_kaggle.yml` so `training.pretrained_ckpt` points at that file,
then re-run the cell above. The cell below does it programmatically.

In [ ]:
import glob, os, re
runs = sorted(glob.glob('/kaggle/working/higanplus/HiGAN+/runs/gan_iam_kaggle-*'))
if runs:
    last_ckpt = os.path.join(runs[-1], 'ckpts', 'last.pth')
    print('latest ckpt :', last_ckpt, '(exists:', os.path.exists(last_ckpt), ')')
    cfg_path = '/kaggle/working/higanplus/HiGAN+/configs/gan_iam_kaggle.yml'
    with open(cfg_path) as f:
        txt = f.read()
    new_txt = re.sub(r"pretrained_ckpt:\s*'[^']*'", f"pretrained_ckpt: '{last_ckpt}'", txt)
    if new_txt != txt:
        with open(cfg_path, 'w') as f:
            f.write(new_txt)
        print('config patched -> resume on next train.py call')
    else:
        print('config unchanged')
else:
    print('no previous kaggle run found')

## 8. Persist outputs

Everything under `/kaggle/working/` is kept as the notebook **Output**. The most useful artifacts:
- `higanplus/HiGAN+/runs/gan_iam_kaggle-*/ckpts/last.pth` — final model (resume-compatible).
- `higanplus/HiGAN+/runs/gan_iam_kaggle-*/imgs/` — sampled handwriting PNGs.
- `higanplus/HiGAN+/runs/gan_iam_kaggle-*/events.out.tfevents.*` — tensorboard.

If you want to keep them as a reusable **Kaggle Dataset**, run:
```python
!cp -r /kaggle/working/higanplus/HiGAN+/runs /kaggle/working/runs_higan_kaggle
```
and then in the Kaggle UI: *Output → Save Version → New Dataset*.

In [ ]:
!du -sh /kaggle/working/higanplus/HiGAN+/runs/* 2>/dev/null || echo 'no runs yet'

## 9. Quick qualitative test on the trained checkpoint

In [ ]:
import glob, os
runs = sorted(glob.glob('/kaggle/working/higanplus/HiGAN+/runs/gan_iam_kaggle-*'))
if runs:
    ckpt = os.path.join(runs[-1], 'ckpts', 'last.pth')
    %cd /kaggle/working/higanplus/HiGAN+
    !python run_demo.py --text "hello kaggle" --out demo_hello.png --ckpt {ckpt}
    from IPython.display import Image, display
    display(Image(filename='/kaggle/working/higanplus/HiGAN+/demo_hello.png'))
else:
    print('train first')